In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6420\2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[sql_query]
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [6]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='6d3f0d44-ddf1-4a00-9b47-7569c1e7b700'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'sql_query', 'arguments': '{"query": "SELECT name, count(*) as track_count\\nFROM artists\\nJOIN albums ON artists.artistid = albums.artistid\\nJOIN tracks ON albums.albumid = tracks.albumid\\nWHERE name LIKE \'S%\'\\nGROUP BY artists.name\\nORDER BY track_count DESC\\nLIMIT 1;"}'}, '__gemini_function_call_thought_signatures__': {'call_14987820113440161938': 'EnMKcQFpFH0TJ/Aby842ASGXlHR2UTW+/hmvPhjQPmaSw+B+6cmxrRluL7gbRIrCPGuFwcBEtXhJi4I78ewFcZewOZ1gXYOxqZPpKlgd+fIGrP30n3Gb+UqcFd0FotYFgv9O/tNY56J2jwNKIiUaCSAqQMwR'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cfdd-60fc-7390-8ba2-868cc3035be7-0', tool_calls=[{'name': 'sql_query', '

In [7]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT Artist.Name, count(*) as track_count
FROM Artist
JOIN Album ON Artist.ArtistId = Album.ArtistId
JOIN Track ON Album.AlbumId = Track.AlbumId
WHERE Artist.Name LIKE 'S%'
GROUP BY Artist.Name
ORDER BY track_count DESC
LIMIT 1;
